# Clustering — Exercises

Companion to the note [Clustering](Clustering.md).

Practise k-means (objective, Lloyd's algorithm, k-means++), k-medoids, hierarchical linkage, DBSCAN, spectral clustering, choosing $k$ and internal/external evaluation (silhouette, Rand/ARI), first by hand and then from scratch in NumPy, checked against scikit-learn and SciPy.

**18 exercises** · 🧠 Concept ×3 · ✍️ By hand ×7 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score, rand_score
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage
rng = np.random.default_rng(0)
# Shared 2-D data: 4 well separated blobs
X_blobs, y_blobs = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=7)

## Part A · Concepts

### Exercise 1 · Pick the algorithm
*🧠 Concept · ★☆☆*

For each data set, which algorithm from the note's table would you try first, and why would plain k-means struggle?

(a) Two interleaving half-moons.  (b) Three round blobs of equal size, $k$ known.  (c) GPS points of taxi pickups with dense hot-spots and scattered noise, number of hot-spots unknown.
(d) You need a taxonomy (clusters inside clusters) of 200 documents.  (e) Two concentric rings.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

k-means assumes convex, roughly spherical, similarly sized clusters and assigns every point to some cluster.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **DBSCAN** or **spectral**: the moons are non-convex; k-means cuts them with a straight line.
(b) **k-means** (with k-means++): this is exactly its assumption.
(c) **DBSCAN** (or HDBSCAN): finds an unknown number of dense regions and labels the rest as **noise**; k-means would force every noise point into a cluster.
(d) **Agglomerative (hierarchical)**: the dendrogram *is* the nested taxonomy; cut it at different heights.
(e) **Spectral clustering** (or DBSCAN): the rings share a centre, so their centroids coincide and k-means fails completely.

</details>

### Exercise 2 · Single vs complete linkage
*🧠 Concept · ★☆☆*

Define single, complete and average linkage distance between clusters $A$ and $B$. Which one suffers from **chaining**
(long, thin clusters connected through a bridge of points) and which one prefers compact clusters of similar diameter?
What does Ward linkage merge?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Single = closest pair, complete = farthest pair.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- Single: $d(A,B)=\min_{a\in A,b\in B}\lVert a-b\rVert$. One close pair is enough to merge, so a chain of points glues clusters together (**chaining**). It can, however, follow non-convex shapes.
- Complete: $\max_{a,b}\lVert a-b\rVert$. Merging is blocked by the farthest pair, which favours **compact** clusters of similar diameter (sensitive to outliers).
- Average: the mean of all pairwise distances, a compromise.
- Ward: merges the pair whose union gives the **smallest increase in within-cluster SSE**, the same objective as k-means $J$, so it gives k-means-like clusters.

</details>

### Exercise 3 · k-means is hard-assignment EM
*🧠 Concept · ★★☆*

The note says k-means is "a hard-assignment special case of EM". Make this precise: start from a GMM with shared covariance
$\sigma^2 I$ and equal weights, and describe what happens to the responsibilities $\gamma_{ik}$ as $\sigma\to 0$.
Which step of Lloyd's algorithm corresponds to the E-step and which to the M-step? See [[Gaussian Mixture Models and EM]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\gamma_{ik}\propto \exp\!\left(-\lVert x_i-\mu_k\rVert^2/(2\sigma^2)\right)$. What does a softmax with temperature $\to 0$ become?

</details>

<details>
<summary><b>✅ Solution</b></summary>

With equal $\pi_k$ and covariance $\sigma^2 I$,
$\gamma_{ik}=\dfrac{\exp(-\lVert x_i-\mu_k\rVert^2/2\sigma^2)}{\sum_j \exp(-\lVert x_i-\mu_j\rVert^2/2\sigma^2)}$, a softmax of $-\lVert x_i-\mu_k\rVert^2$ with temperature $2\sigma^2$.
As $\sigma\to0$ it becomes a one-hot vector on the nearest centroid: the **assignment step** = E-step.
The M-step $\mu_k=\sum_i\gamma_{ik}x_i/\sum_i\gamma_{ik}$ with one-hot $\gamma$ is the cluster mean: the **update step**.
The log-likelihood, scaled by $\sigma^2$, tends to $-\tfrac12 J$, so maximizing it means minimizing the k-means objective.

</details>

## Part B · By hand

### Exercise 4 · One Lloyd iteration
*✍️ By hand · ★☆☆*

1-D data $x=(1,2,4,7,8,10)$, $k=2$, initial centroids $\mu_1=1,\ \mu_2=10$.
Do one assignment step and one update step. Give the new centroids and the objective $J=\sum_i (x_i-\mu_{c(i)})^2$ **after** the update.

In [ ]:
mu_new = None  # [mu1, mu2]
J_new = None

_x = np.array([1, 2, 4, 7, 8, 10.]); _mu = np.array([1., 10.])
_c = np.argmin(np.abs(_x[:, None] - _mu[None]), 1)
_m = np.array([_x[_c == k].mean() for k in range(2)])
check('new centroids', mu_new, _m)
check('J after update', J_new, ((_x - _m[_c]) ** 2).sum())

<details>
<summary><b>💡 Hint 1</b></summary>

4 is at distance 3 from $\mu_1$ and 6 from $\mu_2$; 7 is the other way round.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Clusters $\{1,2,4\}$ and $\{7,8,10\}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Assignment: $\{1,2,4\}\to\mu_1$, $\{7,8,10\}\to\mu_2$. Update: $\mu_1=7/3\approx2.333$, $\mu_2=25/3\approx8.333$.
Within each cluster the deviations are $(-\tfrac43,-\tfrac13,\tfrac53)$, squares sum $=\tfrac{16+1+25}{9}=\tfrac{42}{9}$, so $J=2\cdot\tfrac{42}{9}=\tfrac{28}{3}\approx 9.333$.
A second iteration changes nothing: Lloyd has converged.

```python
mu_new = [7/3, 25/3]
J_new = 28/3
```

</details>

### Exercise 5 · Mean vs medoid with an outlier
*✍️ By hand · ★☆☆*

A cluster contains $\{1,2,3,4,100\}$. Compute the k-means centre (mean) and the k-medoids centre, i.e. the data point
minimizing the sum of absolute distances to the others. Which is the better summary, and why is k-medoids called robust?

In [ ]:
center_mean = None
center_medoid = None

_x = np.array([1, 2, 3, 4, 100.])
check('mean', center_mean, _x.mean())
check('medoid', center_medoid, _x[np.argmin(np.abs(_x[:, None] - _x[None]).sum(1))])

<details>
<summary><b>💡 Hint</b></summary>

Try each candidate point and sum $|x_j - c|$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Mean $=110/5=22$, far from every typical point. Sums of distances: $c=1:105,\ c=2:102,\ c=3:101,\ c=4:102,\ c=100:390$, so the medoid is **3**.
A single outlier drags the mean arbitrarily far but cannot move the medoid much (it must be a real, central point), hence robust.

```python
center_mean = 22.0
center_medoid = 3.0
```

</details>

### Exercise 6 · Silhouette of a single point
*✍️ By hand · ★★☆*

Clusters $A=\{0,1\}$ and $B=\{4,6\}$ on the real line. For point $x=1$ compute
$a$ (mean distance to its own cluster), $b$ (mean distance to the nearest other cluster) and $s=\frac{b-a}{\max(a,b)}$.

In [ ]:
s_point = None

check('silhouette of x=1', s_point, silhouette_samples(np.array([[0.],[1.],[4.],[6.]]), [0,0,1,1])[1])

<details>
<summary><b>💡 Hint</b></summary>

$a$ excludes the point itself: the only other member of $A$ is 0.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$a=|1-0|=1$, $b=\tfrac{|1-4|+|1-6|}{2}=4$, $s=\tfrac{4-1}{4}=0.75$. Values near 1 mean well inside its cluster, near 0 on a border, negative means probably misassigned.

```python
s_point = 0.75
```

</details>

### Exercise 7 · Single-linkage dendrogram heights
*✍️ By hand · ★★☆*

Run agglomerative clustering with **single** linkage on the 1-D points $0,\ 1,\ 3,\ 7,\ 8.5$.
List the four merge heights in the order they happen. What two clusters do you get if you cut the dendrogram at height 3?

In [ ]:
heights = None  # list of 4 numbers

check('merge heights', heights, linkage(np.array([[0],[1],[3],[7],[8.5]]), 'single')[:, 2])

<details>
<summary><b>💡 Hint</b></summary>

Single linkage = distance between the closest members. Start with all pairwise neighbour gaps: 1, 2, 4, 1.5.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Merge $\{0\},\{1\}$ at 1; $\{7\},\{8.5\}$ at 1.5; $\{0,1\},\{3\}$ at 2; finally $\{0,1,3\},\{7,8.5\}$ at $7-3=4$. Heights $(1,1.5,2,4)$.
Cutting at 3 leaves $\{0,1,3\}$ and $\{7,8.5\}$.

```python
heights = [1, 1.5, 2, 4]
```

</details>

### Exercise 8 · Rand index by hand
*✍️ By hand · ★★☆*

True labels $(0,0,0,1,1,1)$, predicted $(0,0,1,1,1,1)$. The Rand index is the fraction of the $\binom{6}{2}=15$ pairs on which
both labelings agree (both "same cluster" or both "different cluster"). Compute it. Why do we usually report the **adjusted** Rand index (ARI) instead?

In [ ]:
RI = None

check('Rand index', RI, rand_score([0,0,0,1,1,1], [0,0,1,1,1,1]))

<details>
<summary><b>💡 Hint</b></summary>

Count same–same pairs (within true clusters) and different–different pairs separately.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Same–same: in true cluster $\{0,1,2\}$ only pair $(0,1)$ stays together (1); in $\{3,4,5\}$ all 3 pairs. Different–different: of the 9 cross pairs, the 6 involving points 0 or 1 stay apart.
$RI=(1+3+6)/15=10/15\approx0.667$.
The RI of a random labelling is not 0 (often well above 0.5), so it is hard to interpret; ARI subtracts the expected value under random labelling, giving 0 for chance and 1 for a perfect match. Both are invariant to permuting label names.

```python
RI = 10/15
```

</details>

### Exercise 9 · DBSCAN by hand
*✍️ By hand · ★★☆*

Points on a line: $0,\ 0.5,\ 1,\ 1.5,\ 5,\ 5.4,\ 9$. Use $\varepsilon=1$ and minPts $=3$ (a point counts itself, as in scikit-learn).
Which points are **core** points? How many clusters are found and how many points are noise?

In [ ]:
n_core = None
n_clusters = None
n_noise = None

_x = np.array([0, .5, 1, 1.5, 5, 5.4, 9]).reshape(-1, 1)
_db = DBSCAN(eps=1, min_samples=3).fit(_x)
check('core points', n_core, len(_db.core_sample_indices_))
check('clusters', n_clusters, len(set(_db.labels_) - {-1}))
check('noise', n_noise, (_db.labels_ == -1).sum())

<details>
<summary><b>💡 Hint</b></summary>

Count the neighbours within distance 1 of each point (including itself). 5 has only 5.4 nearby.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Neighbourhood sizes: $0\!:\!3,\ 0.5\!:\!4,\ 1\!:\!4,\ 1.5\!:\!3,\ 5\!:\!2,\ 5.4\!:\!2,\ 9\!:\!1$.
Core: $0,0.5,1,1.5$ (4 points), forming **one** cluster. $5$ and $5.4$ are not core and not within $\varepsilon$ of a core point, so with 9 there are **3 noise** points.
Note that with minPts $=2$ the pair $\{5,5.4\}$ would become a second cluster.

```python
n_core = 4
n_clusters = 1
n_noise = 3
```

</details>

### Exercise 10 · Lloyd's algorithm never increases J
*✍️ By hand · ★★☆*

Prove that neither step of Lloyd's algorithm can increase $J=\sum_i\lVert x_i-\mu_{c(i)}\rVert^2$, and conclude that it
converges in finitely many steps. Why is the limit only a **local** minimum?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

For fixed assignments, $\sum_{i\in C_k}\lVert x_i-\mu\rVert^2$ is minimized by the mean (set the gradient to zero). For fixed centroids, each point picks its closest one.

</details>

<details>
<summary><b>✅ Solution</b></summary>

*Assignment step*: with $\mu$ fixed, each term $\lVert x_i-\mu_{c(i)}\rVert^2$ is minimized separately by choosing the closest centroid, so $J$ cannot increase.
*Update step*: with $c$ fixed, $\nabla_{\mu_k}\sum_{i\in C_k}\lVert x_i-\mu_k\rVert^2=-2\sum_{i\in C_k}(x_i-\mu_k)=0\Rightarrow\mu_k=\bar x_{C_k}$, the unique minimizer, so $J$ cannot increase.
$J\ge0$ is non-increasing and there are finitely many partitions ($\le k^N$); a partition cannot repeat unless $J$ stays constant, so the algorithm stops after finitely many steps.
It is coordinate descent on a non-convex problem: it stops at a partition no single step improves, which depends on the initialization. Hence k-means++ and several restarts (`n_init`).

</details>

## Part C · Code from scratch

### Exercise 11 · Lloyd's k-means
*💻 Code · ★☆☆*

Implement `kmeans(X, init_centers, n_iter)` returning `(centers, labels, J)`. Run it on `X_blobs` starting from the
first 4 data points as initial centres and compare with `KMeans(init=..., n_init=1)` from scikit-learn.

In [ ]:
def kmeans(X, init_centers, n_iter=100):
    # TODO: alternate assignment (argmin of squared distances) and update (cluster means)
    return None, None, None

C_mine, lab_mine, J_mine = kmeans(X_blobs, X_blobs[:4].copy())

_km = KMeans(4, init=X_blobs[:4].copy(), n_init=1, max_iter=300, tol=1e-10).fit(X_blobs)
check('objective J equals sklearn inertia_', J_mine, _km.inertia_, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Squared distances to all centres at once: `((X[:, None, :] - C[None]) ** 2).sum(-1)` has shape (N, k). Stop when the labels no longer change.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With the same initial centres both implementations follow exactly the same path, so the objectives match. An empty cluster needs special handling: here we keep its old centre (sklearn re-seeds it at a far-away point).

```python
def kmeans(X, init_centers, n_iter=100):
    C = init_centers.astype(float).copy()
    labels = None
    for _ in range(n_iter):
        D = ((X[:, None, :] - C[None]) ** 2).sum(-1)
        new = D.argmin(1)
        if labels is not None and np.array_equal(new, labels):
            break
        labels = new
        # keep the old centre if a cluster becomes empty
        C = np.array([X[labels == k].mean(0) if (labels == k).any() else C[k] for k in range(len(C))])
    J = ((X - C[labels]) ** 2).sum()
    return C, labels, J

C_mine, lab_mine, J_mine = kmeans(X_blobs, X_blobs[:4].copy())
```

</details>

### Exercise 12 · k-means++ initialization
*💻 Code · ★★☆*

Implement `kmeanspp(X, k, rng)`: the first centre is a uniformly random data point; each next centre is a data point drawn
with probability proportional to $D(x)^2$, the squared distance to the nearest centre chosen so far.
Then compare the final $J$ of your `kmeans` started from 20 random-point initializations vs. 20 k-means++ initializations
(store the mean final $J$ of each in `J_random`, `J_pp`).

In [ ]:
def kmeanspp(X, k, rng):
    # TODO: return a (k, d) array of chosen data points
    return None

J_random = None
J_pp = None

_C = kmeanspp(X_blobs, 4, np.random.default_rng(3))
check('k-means++ returns 4 distinct data points', None if _C is None else (len({tuple(c) for c in _C}) == 4 and all((np.abs(X_blobs - c).sum(1) < 1e-12).any() for c in _C)), True)
check('k-means++ is not worse on average', None if J_pp is None else J_pp <= J_random + 1e-9, True)

<details>
<summary><b>💡 Hint</b></summary>

`d2 = ((X[:, None] - C[None]) ** 2).sum(-1).min(1)`, then `rng.choice(len(X), p=d2 / d2.sum())`. Already chosen points have $D=0$ and cannot be picked again.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Random initialization sometimes puts two centres in the same blob, and Lloyd then gets stuck in a poor local minimum (one centre splitting a blob while another covers two).
k-means++ spreads the centres out; it is $O(\log k)$-competitive with the optimal $J$ in expectation, and on these blobs it essentially always finds the global optimum.

```python
def kmeanspp(X, k, rng):
    C = [X[rng.integers(len(X))]]
    for _ in range(1, k):
        d2 = ((X[:, None] - np.array(C)[None]) ** 2).sum(-1).min(1)
        C.append(X[rng.choice(len(X), p=d2 / d2.sum())])
    return np.array(C)

_r = np.random.default_rng(7)
J_random = np.mean([kmeans(X_blobs, X_blobs[_r.choice(len(X_blobs), 4, replace=False)])[2] for _ in range(20)])
J_pp = np.mean([kmeans(X_blobs, kmeanspp(X_blobs, 4, _r))[2] for _ in range(20)])
print(f"mean J random init: {J_random:.1f}   mean J k-means++: {J_pp:.1f}")
```

</details>

### Exercise 13 · Choosing k: elbow and silhouette
*💻 Code · ★★☆*

For $k=2,\dots,8$ fit `KMeans(k, n_init=10, random_state=0)` on `X_blobs`, record the inertia (SSE) and the silhouette score.
Plot both against $k$ and store the $k$ with the highest silhouette in `best_k`. Is the elbow at the same place?

In [ ]:
import matplotlib.pyplot as plt
ks = list(range(2, 9))
best_k = None

check('best k by silhouette', best_k, 4)

<details>
<summary><b>💡 Hint</b></summary>

`silhouette_score(X, labels)`. The elbow is where the SSE curve stops dropping steeply.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both criteria point to $k=4$, the true number of blobs. SSE always decreases with $k$ (with $k=N$ it is 0), so we look for a kink rather than a minimum; silhouette has a genuine maximum. The gap statistic or BIC of a GMM are alternatives from the note.

```python
import matplotlib.pyplot as plt
ks = list(range(2, 9))
sse, sil = [], []
for k in ks:
    km = KMeans(k, n_init=10, random_state=0).fit(X_blobs)
    sse.append(km.inertia_)
    sil.append(silhouette_score(X_blobs, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(ks, sse, 'o-'); ax[0].set_title('SSE (elbow)')
ax[1].plot(ks, sil, 'o-'); ax[1].set_title('silhouette')
plt.show()
best_k = ks[int(np.argmax(sil))]
```

</details>

### Exercise 14 · Scale your features first
*💻 Code · ★☆☆*

The data below has two informative features, but the second is measured in different units (multiplied by 1000) and a third,
pure-noise feature is added on a large scale. Cluster with k-means ($k=3$) **before** and **after** `StandardScaler`, and store the ARI with the true labels in `ari_raw` and `ari_scaled`.

In [ ]:
X3, y3 = make_blobs(n_samples=300, centers=[[0, 0], [4, 0], [2, 3.5]], cluster_std=0.7, random_state=2)
X_units = np.c_[X3[:, 0], 1000 * X3[:, 1], rng.normal(scale=50, size=300)]
ari_raw = None
ari_scaled = None

check('scaling helps a lot', None if ari_raw is None else (ari_scaled > 0.8 and ari_scaled > ari_raw + 0.3), True)

<details>
<summary><b>💡 Hint</b></summary>

Euclidean distance is dominated by the feature with the largest numeric range.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Unscaled, the distances are dominated by the $\times1000$ feature, so the first feature is ignored and clusters that differ only in it get merged (ARI around 0.5).
After standardization all features count equally and the ARI is close to 1, even with the noise feature. See [[Data Preprocessing]].

```python
X3, y3 = make_blobs(n_samples=300, centers=[[0, 0], [4, 0], [2, 3.5]], cluster_std=0.7, random_state=2)
X_units = np.c_[X3[:, 0], 1000 * X3[:, 1], rng.normal(scale=50, size=300)]
ari_raw = adjusted_rand_score(y3, KMeans(3, n_init=10, random_state=0).fit_predict(X_units))
ari_scaled = adjusted_rand_score(y3, KMeans(3, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(X_units)))
print(ari_raw, ari_scaled)
```

</details>

### Exercise 15 · DBSCAN from scratch
*💻 Code · ★★★*

Implement `dbscan(X, eps, min_pts)` returning labels (noise $=-1$): find core points (at least `min_pts` neighbours within
`eps`, counting itself), grow clusters by breadth-first search from unvisited core points, attach border points.
Run it on two moons and compare to `sklearn.cluster.DBSCAN` (labels may be permuted, so compare with ARI).

In [ ]:
def dbscan(X, eps, min_pts):
    # TODO
    return None

X_moons, y_moons = make_moons(300, noise=0.06, random_state=0)
lab_db = dbscan(X_moons, eps=0.2, min_pts=5)

check('same clustering as sklearn (ARI)', None if lab_db is None else adjusted_rand_score(DBSCAN(eps=0.2, min_samples=5).fit_predict(X_moons), lab_db), 1.0)

<details>
<summary><b>💡 Hint 1</b></summary>

Precompute the neighbour lists from the full distance matrix: `nbrs = [np.flatnonzero(row <= eps) for row in D]`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Only core points expand the queue; border points get the cluster label but are not expanded.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Complexity is $O(N^2)$ with a full distance matrix ($O(N\log N)$ with a spatial index). Border points reachable from two clusters may get either label (order dependent), which is why we compare with ARI rather than exact labels.
Compare: `KMeans(2)` on the moons gives an ARI of only about 0.2.

```python
def dbscan(X, eps, min_pts):
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    nbrs = [np.flatnonzero(row <= eps) for row in D]
    core = np.array([len(n) >= min_pts for n in nbrs])
    labels = np.full(len(X), -1)
    c = 0
    for i in range(len(X)):
        if labels[i] != -1 or not core[i]:
            continue
        labels[i] = c
        queue = [i]
        while queue:
            j = queue.pop()
            for n in nbrs[j]:
                if labels[n] == -1:
                    labels[n] = c
                    if core[n]:
                        queue.append(n)
        c += 1
    return labels

X_moons, y_moons = make_moons(300, noise=0.06, random_state=0)
lab_db = dbscan(X_moons, eps=0.2, min_pts=5)
print('ARI vs truth:', adjusted_rand_score(y_moons, lab_db))
```

</details>

### Exercise 16 · Spectral clustering from scratch
*💻 Code · ★★★*

Implement normalized spectral clustering (Ng–Jordan–Weiss) on two concentric circles:

1. Affinity $W_{ij}=\exp(-\lVert x_i-x_j\rVert^2/(2s^2))$, $W_{ii}=0$, with $s=0.1$.
2. $L_{sym}=I-D^{-1/2}WD^{-1/2}$ with degree matrix $D$.
3. Take the eigenvectors of the $k=2$ **smallest** eigenvalues, normalize each row to unit length, run k-means on the rows.

Store the ARI with the true labels in `ari_spectral` and the ARI of plain k-means in `ari_kmeans_circles`.

In [ ]:
X_circ, y_circ = make_circles(300, factor=0.4, noise=0.04, random_state=0)
def spectral(X, k=2, s=0.1):
    # TODO
    return None

ari_spectral = None
ari_kmeans_circles = None

check('spectral separates the rings', None if ari_spectral is None else (ari_spectral > 0.95 and ari_kmeans_circles < 0.1), True)

<details>
<summary><b>💡 Hint 1</b></summary>

`np.linalg.eigh` returns eigenvalues in ascending order.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

In the spectral embedding each ring collapses to (almost) a single point, so k-means there is trivial.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The similarity graph has two (nearly) disconnected components, one per ring. A graph Laplacian has one zero eigenvalue per connected component, with eigenvectors that are (scaled) indicator vectors of the components.
So the embedding maps each ring to one point and k-means separates them perfectly, while plain k-means on the raw coordinates splits the plane in half (ARI $\approx0$). The bandwidth $s$ matters: too large connects the rings.

```python
X_circ, y_circ = make_circles(300, factor=0.4, noise=0.04, random_state=0)
def spectral(X, k=2, s=0.1):
    D2 = ((X[:, None] - X[None]) ** 2).sum(-1)
    W = np.exp(-D2 / (2 * s ** 2)); np.fill_diagonal(W, 0)
    d = W.sum(1)
    L = np.eye(len(X)) - W / np.sqrt(np.outer(d, d))
    vals, vecs = np.linalg.eigh(L)
    U = vecs[:, :k]
    U = U / np.linalg.norm(U, axis=1, keepdims=True)
    return KMeans(k, n_init=10, random_state=0).fit_predict(U)

ari_spectral = adjusted_rand_score(y_circ, spectral(X_circ))
ari_kmeans_circles = adjusted_rand_score(y_circ, KMeans(2, n_init=10, random_state=0).fit_predict(X_circ))
print(ari_spectral, ari_kmeans_circles)
```

</details>

### Exercise 17 · Linkage on elongated clusters
*💻 Code · ★★☆*

Fit `AgglomerativeClustering(n_clusters=2, linkage=...)` with `'single'`, `'complete'`, `'average'` and `'ward'` on the two moons
(`X_moons` from Exercise 15) and on `X_blobs` with 4 clusters. Store the ARIs in dicts `ari_moons` and `ari_blobs` keyed by linkage.
Which linkage wins on which data set, and why?

In [ ]:
X_moons, y_moons = make_moons(300, noise=0.06, random_state=0)
links = ['single', 'complete', 'average', 'ward']
ari_moons = None
ari_blobs = None

check('single linkage best on moons', None if ari_moons is None else max(ari_moons, key=ari_moons.get) == 'single', True)
check('ward perfect-ish on blobs', None if ari_blobs is None else ari_blobs['ward'] > 0.95, True)

<details>
<summary><b>💡 Hint</b></summary>

Single linkage can follow a curved chain of nearby points; Ward minimizes within-cluster SSE like k-means.

</details>

<details>
<summary><b>✅ Solution</b></summary>

On the moons, single linkage recovers both half-moons (ARI $\approx1$) because the points within a moon form a chain of small gaps, while complete/average/Ward produce compact, convex pieces.
On these well separated round blobs every linkage is perfect; with noisier, touching blobs single linkage is the first to fail, because a few stray points between blobs form a bridge. The chaining weakness of single linkage is a strength on non-convex shapes.

```python
X_moons, y_moons = make_moons(300, noise=0.06, random_state=0)
links = ['single', 'complete', 'average', 'ward']
ari_moons = {l: adjusted_rand_score(y_moons, AgglomerativeClustering(2, linkage=l).fit_predict(X_moons)) for l in links}
ari_blobs = {l: adjusted_rand_score(y_blobs, AgglomerativeClustering(4, linkage=l).fit_predict(X_blobs)) for l in links}
print(ari_moons); print(ari_blobs)
```

</details>

### Exercise 18 · Silhouette from scratch
*💻 Code · ★★☆*

Implement `silhouette(X, labels)` returning the mean silhouette coefficient, computing $a(i)$ and $b(i)$ from the
pairwise distance matrix (singletons get $s=0$). Check it against `silhouette_score` on the $k=4$ k-means labels of `X_blobs`.

In [ ]:
def silhouette(X, labels):
    # TODO
    return None

lab4 = KMeans(4, n_init=10, random_state=0).fit_predict(X_blobs)
sil_mine = silhouette(X_blobs, lab4)

check('mean silhouette', sil_mine, silhouette_score(X_blobs, lab4))

<details>
<summary><b>💡 Hint</b></summary>

For $a(i)$ divide by $|C_i|-1$ (the point itself has distance 0). For $b(i)$ take the minimum over other clusters of the mean distance.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Silhouette costs $O(N^2)$ distances, so on large data sets it is usually computed on a random subsample (`sample_size=` in sklearn).

```python
def silhouette(X, labels):
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    labels = np.asarray(labels); ks = np.unique(labels)
    s = np.zeros(len(X))
    for i in range(len(X)):
        own = labels == labels[i]
        if own.sum() == 1:
            continue
        a = D[i, own].sum() / (own.sum() - 1)
        b = min(D[i, labels == k].mean() for k in ks if k != labels[i])
        s[i] = (b - a) / max(a, b)
    return s.mean()

lab4 = KMeans(4, n_init=10, random_state=0).fit_predict(X_blobs)
sil_mine = silhouette(X_blobs, lab4)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Clustering](Clustering.md).*